# 🎯 FINAL: Bleeding Prediction MLP Training - Part 2

## Continuation from Part 1

This notebook continues from Part 1 with:
- Training loop with class imbalance handling
- Evaluation metrics
- Comprehensive visualizations
- Final model analysis

**Run Part 1 first to load data and create model!**

---
# 🏋️ SECTION 7: Training Loop
---

## Class Imbalance Handling

Our dataset is **imbalanced**: ~10% positive, ~90% negative

### Solutions:

1. **Weighted Loss**: Give more importance to minority class
   ```python
   weight_positive = total_samples / (2 * positive_samples)
   weight_negative = total_samples / (2 * negative_samples)
   ```

2. **Weighted Sampling**: Sample more positive examples during training

3. **Evaluation Metrics**: Focus on precision, recall, F1 (not just accuracy)

## Training Strategy:

- **Optimizer**: Adam (adaptive learning rate)
- **Learning Rate**: 1e-3 with ReduceLROnPlateau
- **Early Stopping**: Stop if validation loss doesn't improve for 15 epochs
- **Batch Size**: 256 (good for this dataset size)

In [ ]:
# Cell 14: Create PyTorch Datasets and DataLoaders
class BleedingDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create datasets
train_dataset = BleedingDataset(X_train_scaled, y_train)
val_dataset = BleedingDataset(X_val_scaled, y_val)
test_dataset = BleedingDataset(X_test_scaled, y_test)

# Calculate class weights for imbalanced data
class_counts = np.bincount(y_train)
class_weights = len(y_train) / (2 * class_counts)
class_weights_tensor = torch.FloatTensor(class_weights).to(Config.DEVICE)

print(f"\nClass weights for loss function:")
print(f"  Class 0 (no bleeding): {class_weights[0]:.3f}")
print(f"  Class 1 (bleeding): {class_weights[1]:.3f}")
print(f"  Ratio: 1:{class_weights[1]/class_weights[0]:.1f}")

# Create weighted sampler for training (optional - helps with imbalance)
sample_weights = np.array([class_weights[label] for label in y_train])
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=Config.BATCH_SIZE,
    sampler=sampler,  # Use weighted sampler
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\n✅ DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
# Cell 15: Training Setup
# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=Config.LEARNING_RATE,
    weight_decay=Config.WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    verbose=True
)

print("Training setup complete:")
print(f"  Loss: CrossEntropyLoss with class weights")
print(f"  Optimizer: Adam (lr={Config.LEARNING_RATE}, wd={Config.WEIGHT_DECAY})")
print(f"  Scheduler: ReduceLROnPlateau (patience=5, factor=0.5)")
print(f"  Early stopping patience: {Config.EARLY_STOPPING_PATIENCE} epochs")

In [ ]:
# Cell 16: Training and Validation Functions
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for features, labels in tqdm(loader, desc="Training", leave=False):
        features, labels = features.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item() * len(labels)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, accuracy

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for features, labels in tqdm(loader, desc="Validating", leave=False):
            features, labels = features.to(device), labels.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of class 1
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    
    return avg_loss, accuracy, precision, recall, f1, all_preds, all_labels, all_probs

print("✅ Training functions defined")

In [ ]:
# Cell 17: Training Loop
print("\n" + "=" * 80)
print("STARTING TRAINING")
print("=" * 80)
print()

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_precision': [],
    'val_recall': [],
    'val_f1': [],
    'learning_rates': []
}

best_val_loss = float('inf')
best_val_f1 = 0
patience_counter = 0

for epoch in range(Config.NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{Config.NUM_EPOCHS}")
    print("-" * 80)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, Config.DEVICE)
    
    # Validate
    val_loss, val_acc, val_prec, val_rec, val_f1, _, _, _ = validate_epoch(
        model, val_loader, criterion, Config.DEVICE
    )
    
    # Update scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_precision'].append(val_prec)
    history['val_recall'].append(val_rec)
    history['val_f1'].append(val_f1)
    history['learning_rates'].append(current_lr)
    
    # Print metrics
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Val Precision: {val_prec:.4f} | Val Recall: {val_rec:.4f} | Val F1: {val_f1:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")
    
    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_f1': val_f1,
            'history': history,
        }, Config.OUTPUT_DIR / 'best_model.pth')
        print(f"✅ New best model saved! (F1: {val_f1:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= Config.EARLY_STOPPING_PATIENCE:
        print(f"\n⚠️  Early stopping triggered after {epoch+1} epochs")
        print(f"   Best Val F1: {best_val_f1:.4f} at epoch {epoch+1-patience_counter}")
        break

print("\n" + "=" * 80)
print("TRAINING COMPLETE!")
print("=" * 80)
print(f"\nBest Validation F1: {best_val_f1:.4f}")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Total epochs trained: {len(history['train_loss'])}")

# Save final history
with open(Config.OUTPUT_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print(f"\n✅ Training history saved: {Config.OUTPUT_DIR / 'training_history.json'}")

---
# 📈 SECTION 8: Evaluation & Metrics
---

In [ ]:
# Cell 18: Load Best Model and Evaluate on Test Set
print("\n" + "=" * 80)
print("FINAL EVALUATION ON TEST SET")
print("=" * 80)
print()

# Load best model
checkpoint = torch.load(Config.OUTPUT_DIR / 'best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✅ Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"   Validation F1: {checkpoint['val_f1']:.4f}")

# Evaluate on test set
test_loss, test_acc, test_prec, test_rec, test_f1, test_preds, test_labels, test_probs = validate_epoch(
    model, test_loader, criterion, Config.DEVICE
)

# Calculate additional metrics
test_auc = roc_auc_score(test_labels, test_probs)

print(f"\n📊 TEST SET RESULTS:")
print("=" * 80)
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_acc:.4f} ({100*test_acc:.2f}%)")
print(f"  Precision: {test_prec:.4f}")
print(f"  Recall: {test_rec:.4f}")
print(f"  F1-Score: {test_f1:.4f}")
print(f"  AUC-ROC: {test_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)
print(f"\n  Confusion Matrix:")
print(f"                Predicted")
print(f"              No    Yes")
print(f"  Actual No  {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"        Yes  {cm[1,0]:5d}  {cm[1,1]:5d}")

# Classification report
print(f"\n  Detailed Classification Report:")
print(classification_report(test_labels, test_preds, 
                          target_names=['No Bleeding', 'Bleeding'],
                          digits=4))

# Save results
results = {
    'test_loss': float(test_loss),
    'test_accuracy': float(test_acc),
    'test_precision': float(test_prec),
    'test_recall': float(test_rec),
    'test_f1': float(test_f1),
    'test_auc': float(test_auc),
    'confusion_matrix': cm.tolist(),
}

with open(Config.OUTPUT_DIR / 'test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Test results saved: {Config.OUTPUT_DIR / 'test_results.json'}")

---
# 📊 SECTION 9: Visualizations
---

In [ ]:
# Cell 19: Training History Plots
print("Creating training history visualizations...\n")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

epochs = range(1, len(history['train_loss']) + 1)

# 1. Loss
axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Validation', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Accuracy
axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train', linewidth=2)
axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Validation', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy', fontsize=12)
axes[0, 1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision, Recall, F1
axes[0, 2].plot(epochs, history['val_precision'], 'g-', label='Precision', linewidth=2)
axes[0, 2].plot(epochs, history['val_recall'], 'b-', label='Recall', linewidth=2)
axes[0, 2].plot(epochs, history['val_f1'], 'r-', label='F1-Score', linewidth=2)
axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('Score', fontsize=12)
axes[0, 2].set_title('Validation Metrics', fontsize=14, fontweight='bold')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. Learning Rate
axes[1, 0].plot(epochs, history['learning_rates'], 'purple', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# 5. Confusion Matrix (Test Set)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
           xticklabels=['No Bleeding', 'Bleeding'],
           yticklabels=['No Bleeding', 'Bleeding'])
axes[1, 1].set_xlabel('Predicted', fontsize=12)
axes[1, 1].set_ylabel('Actual', fontsize=12)
axes[1, 1].set_title('Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')

# 6. ROC Curve
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
axes[1, 2].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {test_auc:.4f}')
axes[1, 2].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[1, 2].set_xlabel('False Positive Rate', fontsize=12)
axes[1, 2].set_ylabel('True Positive Rate', fontsize=12)
axes[1, 2].set_title('ROC Curve (Test Set)', fontsize=14, fontweight='bold')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / '03_training_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Training evaluation plots saved")

In [ ]:
# Cell 20: Prediction Examples on Test Videos
print("\nCreating prediction examples...\n")

# Get predictions for test videos
test_video_list = split_info['test_videos'][:2]  # Show 2 test videos

for video_id in test_video_list:
    print(f"Processing {video_id}...")
    
    # Get video data
    video_data = all_data[video_id]
    
    # Create features
    features, valid_indices = create_temporal_features(
        video_data,
        window_size=Config.WINDOW_SIZE,
        include_temporal_changes=True
    )
    
    # Scale features
    features_scaled = scaler.transform(features)
    
    # Get predictions
    model.eval()
    with torch.no_grad():
        features_tensor = torch.FloatTensor(features_scaled).to(Config.DEVICE)
        outputs = model(features_tensor)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()  # P(bleeding)
        preds = (probs > 0.5).astype(int)
    
    # Get true labels
    true_labels = create_labels(
        valid_indices,
        video_data['peaks'],
        prediction_horizon=Config.PREDICTION_HORIZON
    )
    
    # Plot
    fig, axes = plt.subplots(3, 1, figsize=(20, 10), sharex=True)
    
    frames = video_data['frame_indices']
    blood = video_data['smoothed_blood_areas']
    peaks = video_data['peaks']
    
    # 1. Blood area
    axes[0].plot(frames, blood, color='darkred', linewidth=2)
    axes[0].fill_between(frames, 0, blood, alpha=0.2, color='red')
    if len(peaks) > 0:
        axes[0].scatter(frames[peaks], blood[peaks], 
                       color='red', s=150, marker='*', zorder=5,
                       edgecolors='black', linewidths=1.5,
                       label=f'True Peaks ({len(peaks)})')
    axes[0].set_ylabel('Blood Area', fontsize=12, fontweight='bold')
    axes[0].set_title(f'{video_id.upper()} - Blood Area & Predictions', 
                     fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. True labels
    axes[1].fill_between(valid_indices, 0, true_labels, 
                        alpha=0.6, color='green', label='True: Bleeding Will Occur')
    axes[1].set_ylabel('True Label', fontsize=12, fontweight='bold')
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].set_yticks([0, 1])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # 3. Predicted probabilities
    axes[2].plot(valid_indices, probs, color='blue', linewidth=2, label='P(Bleeding)')
    axes[2].axhline(0.5, color='red', linestyle='--', label='Threshold')
    axes[2].fill_between(valid_indices, 0, probs, 
                        where=(probs > 0.5), alpha=0.3, color='orange',
                        label='Predicted: Bleeding Will Occur')
    axes[2].set_xlabel('Frame Index', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('Probability', fontsize=12, fontweight='bold')
    axes[2].set_ylim(-0.05, 1.05)
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(Config.OUTPUT_DIR / f'04_predictions_{video_id}.png', 
               dpi=150, bbox_inches='tight')
    plt.show()
    
    # Calculate metrics for this video
    video_acc = accuracy_score(true_labels, preds)
    video_prec = precision_score(true_labels, preds, zero_division=0)
    video_rec = recall_score(true_labels, preds, zero_division=0)
    video_f1 = f1_score(true_labels, preds, zero_division=0)
    
    print(f"  Accuracy: {video_acc:.4f}")
    print(f"  Precision: {video_prec:.4f}")
    print(f"  Recall: {video_rec:.4f}")
    print(f"  F1-Score: {video_f1:.4f}")
    print()

print(f"✅ Prediction visualizations saved")

---
# 📝 SECTION 10: Final Summary
---

In [ ]:
# Cell 21: Create Final Summary Report
summary_text = f"""BLEEDING PREDICTION MLP - FINAL SUMMARY
{'=' * 80}

DATASET
{'-' * 80}
Total Videos: {len(all_data)}
Total Frames: {len(X):,}
Total Bleeding Events: {sum([len(d['peaks']) for d in all_data.values()])}

Split:
  Train: {len(train_videos)} videos, {len(X_train):,} samples ({100*len(X_train)/len(X):.1f}%)
  Val:   {len(val_videos)} videos, {len(X_val):,} samples ({100*len(X_val)/len(X):.1f}%)
  Test:  {len(test_videos)} videos, {len(X_test):,} samples ({100*len(X_test)/len(X):.1f}%)

Class Distribution:
  Positive (bleeding will occur): {np.sum(y):,} ({100*np.mean(y):.2f}%)
  Negative (no bleeding): {np.sum(y==0):,} ({100*np.mean(y==0):.2f}%)
  Imbalance ratio: 1:{np.sum(y==0) / max(np.sum(y), 1):.1f}

FEATURES
{'-' * 80}
Window Size: {Config.WINDOW_SIZE} frames
Prediction Horizon: {Config.PREDICTION_HORIZON} frames ahead
Feature Dimension: {X.shape[1]}
  - Basic features: {Config.WINDOW_SIZE * Config.NUM_BASIC_FEATURES}
  - Temporal changes: {X.shape[1] - Config.WINDOW_SIZE * Config.NUM_BASIC_FEATURES}

MODEL ARCHITECTURE
{'-' * 80}
Type: Multi-Layer Perceptron (MLP)
Hidden Layers: {len(Config.HIDDEN_DIMS)}
Hidden Dimensions: {Config.HIDDEN_DIMS}
Total Parameters: {total_params:,}
Dropout Rates: {Config.DROPOUT_RATES}

TRAINING
{'-' * 80}
Optimizer: Adam (lr={Config.LEARNING_RATE}, wd={Config.WEIGHT_DECAY})
Loss: CrossEntropyLoss with class weights
Batch Size: {Config.BATCH_SIZE}
Epochs Trained: {len(history['train_loss'])}
Early Stopping: {'Yes' if patience_counter >= Config.EARLY_STOPPING_PATIENCE else 'No'}

RESULTS - TEST SET
{'-' * 80}
Accuracy:  {test_acc:.4f} ({100*test_acc:.2f}%)
Precision: {test_prec:.4f}
Recall:    {test_rec:.4f}
F1-Score:  {test_f1:.4f}
AUC-ROC:   {test_auc:.4f}

Confusion Matrix:
              Predicted
            No    Yes
Actual No  {cm[0,0]:5d}  {cm[0,1]:5d}
      Yes  {cm[1,0]:5d}  {cm[1,1]:5d}

INTERPRETATION
{'-' * 80}
The model predicts bleeding events {Config.PREDICTION_HORIZON} frames ({Config.PREDICTION_HORIZON} seconds at 1 FPS) 
before they occur with:
  - {100*test_prec:.1f}% precision (when it says bleeding will occur, it's right {100*test_prec:.1f}% of the time)
  - {100*test_rec:.1f}% recall (it catches {100*test_rec:.1f}% of actual bleeding events)
  - {100*test_f1:.1f}% F1-score (balanced performance)

This gives surgeons approximately {Config.PREDICTION_HORIZON} seconds of warning before bleeding occurs,
allowing for preventive action.

OUTPUT FILES
{'-' * 80}
{Config.OUTPUT_DIR}/
  ├── best_model.pth                  ← Trained model weights
  ├── scaler.pkl                      ← Feature scaler
  ├── training_history.json           ← Training metrics
  ├── test_results.json               ← Test set results
  ├── split_info.json                 ← Train/val/test split
  ├── 01_data_distribution.png        ← Data visualizations
  ├── 02_labeling_strategy.png        ← Label generation
  ├── 03_training_evaluation.png      ← Training curves
  └── 04_predictions_*.png            ← Example predictions

NEXT STEPS
{'-' * 80}
1. Deploy model for real-time inference
2. Collect more diverse surgical videos
3. Fine-tune on specific surgical procedures
4. Integrate with surgical visualization systems
5. Prepare manuscript for IEEE ISBI 2026

{'=' * 80}
"""

print(summary_text)

# Save summary
with open(Config.OUTPUT_DIR / 'FINAL_SUMMARY.txt', 'w') as f:
    f.write(summary_text)

print(f"\n✅ Final summary saved: {Config.OUTPUT_DIR / 'FINAL_SUMMARY.txt'}")

In [ ]:
# Cell 22: Print Final Message
print("\n" + "=" * 80)
print("🎉 CONGRATULATIONS! BLEEDING PREDICTION MODEL COMPLETE!")
print("=" * 80)
print()
print("✅ Model successfully trained and evaluated")
print("✅ All visualizations generated")
print("✅ Results documented")
print()
print(f"📊 Final Test F1-Score: {test_f1:.4f}")
print(f"📈 Model can predict bleeding {Config.PREDICTION_HORIZON} frames ahead")
print()
print("📁 All outputs saved to:", Config.OUTPUT_DIR)
print()
print("🚀 Ready for deployment and IEEE ISBI 2026 submission!")
print("=" * 80)